# Clever Sydney GGUF on Kaggle T4 x2

把 `FPHam/Clever_Sydney-4_12b_GGUF` 部署成临时 OpenAI-compatible API。

Kaggle 设置：
- Accelerator: **GPU T4 x2**
- Internet: **On**

运行后会输出一个 Cloudflare Tunnel URL，填到本地 `.env`：

```env
TEACHER_BASE_URL=https://xxxx.trycloudflare.com/v1
TEACHER_MODEL=clever-sydney-4-12b-q8
TEACHER_API_PROTOCOL=legacy_chat_completions
```


In [ ]:
# 1) 环境检查
!nvidia-smi
import os, subprocess, textwrap, time, json, re, pathlib, shutil
print('Kaggle working dir:', os.getcwd())


In [ ]:
# 2) Install llama-cpp-python CUDA server on Kaggle
# Goal: avoid CPU fallback. We install dependencies from PyPI first, then install llama-cpp-python itself
# from the CUDA wheel index with --no-deps and WITHOUT PyPI fallback.
import os, sys, subprocess, pathlib, shutil, re, importlib
from pathlib import Path

WORK = Path('/kaggle/working')
BIN_DIR = WORK / 'bin'
BIN_DIR.mkdir(exist_ok=True, parents=True)
server_cmd = None


def run(cmd, check=True, capture=False):
    print('\n$ ' + cmd)
    p = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    if capture:
        print((p.stdout or '')[-12000:])
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd, output=getattr(p, 'stdout', None))
    return p


def clear_llama_modules():
    for name in list(sys.modules):
        if name == 'llama_cpp' or name.startswith('llama_cpp.'):
            del sys.modules[name]


def detect_cuda():
    try:
        import torch
        print('torch CUDA:', torch.version.cuda)
        print('cuda available:', torch.cuda.is_available(), 'gpu count:', torch.cuda.device_count())
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                print(i, torch.cuda.get_device_name(i))
    except Exception as e:
        print('torch CUDA detection failed:', repr(e))

detect_cuda()

# Clean previous CPU install. If you imported llama_cpp before, a kernel restart is safest.
run(f'{sys.executable} -m pip uninstall -y llama-cpp-python llama_cpp_python llama-cpp-python-cuda || true', check=False)
clear_llama_modules()

# Dependencies for the OpenAI-compatible server. Keep this separate from the CUDA wheel install.
run(f'{sys.executable} -m pip install -q --upgrade pip setuptools wheel')
run(
    f'{sys.executable} -m pip install -q --upgrade '
    'huggingface_hub requests fastapi "uvicorn[standard]" sse-starlette pydantic-settings starlette-context'
)

# Kaggle may report CUDA 12.8/12.9, but cu124 wheels are usually the most compatible available public wheels.
# Use --index-url and --no-deps so pip cannot silently pick a CPU build from PyPI.
CUDA_WHEEL_TAG = 'cu124'
LLAMA_CPP_VERSION = '0.3.23'
run(
    f'{sys.executable} -m pip install -q --no-cache-dir --force-reinstall --no-deps '
    f'llama-cpp-python=={LLAMA_CPP_VERSION} '
    f'--index-url https://abetlen.github.io/llama-cpp-python/whl/{CUDA_WHEEL_TAG}',
    capture=True,
)
clear_llama_modules()

import llama_cpp
print('llama-cpp-python:', llama_cpp.__version__)
print('llama_cpp module:', llama_cpp.__file__)
try:
    gpu_ok = bool(llama_cpp.llama_supports_gpu_offload())
except Exception as e:
    gpu_ok = False
    print('GPU check exception:', repr(e))
print('GPU offload supported:', gpu_ok)

if not gpu_ok:
    raise RuntimeError(
        'llama-cpp-python is still CPU-only. Restart Kaggle Session, rerun this cell, and ensure it installs from cu124. '
        'Do not use --extra-index-url for llama-cpp-python itself.'
    )

server_cmd = sys.executable + ' -m llama_cpp.server'
print('SERVER_CMD=', server_cmd)


In [ ]:
# 3) 下载 GGUF 到 /kaggle/working/models
from huggingface_hub import hf_hub_download
from pathlib import Path

MODEL_REPO = 'FPHam/Clever_Sydney-4_12b_GGUF'
MODEL_FILE = 'Clever_Sydney-4_12b_Q8_0_o.gguf'
MODEL_DIR = Path('/kaggle/working/models')
MODEL_DIR.mkdir(exist_ok=True, parents=True)
MODEL_PATH = MODEL_DIR / MODEL_FILE

if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size < 1_000_000_000:
    p = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=str(MODEL_DIR), local_dir_use_symlinks=False, resume_download=True)
    print('downloaded:', p)
else:
    print('model exists:', MODEL_PATH, MODEL_PATH.stat().st_size / 1e9, 'GB')

print('MODEL_PATH=', MODEL_PATH)


In [ ]:
# 4) Start OpenAI-compatible server
# Robust version: build the command from scratch and assert there is no old "1,1" token.
import os, subprocess, time, requests, shlex, sys, pathlib

PORT = 8000
SERVED_MODEL_NAME = 'clever-sydney-4-12b-q8'
LOG_PATH = '/kaggle/working/sydney_server.log'

# Stop old server if this cell was run before.
try:
    if 'server_proc' in globals() and server_proc and server_proc.poll() is None:
        print('Stopping previous server...')
        server_proc.terminate()
        time.sleep(3)
        if server_proc.poll() is None:
            server_proc.kill()
except Exception as e:
    print('old server cleanup skipped:', repr(e))

# IMPORTANT for llama-cpp-python:
#   Correct:   --tensor_split 1 1
#   Incorrect: --tensor_split 1,1
# If your printed command still shows "--tensor_split 1,1", you are running an old cell.
cmd = [
    sys.executable, '-m', 'llama_cpp.server',
    '--model', str(MODEL_PATH),
    '--model_alias', SERVED_MODEL_NAME,
    '--host', '0.0.0.0',
    '--port', str(PORT),
    '--n_gpu_layers', '-1',
    '--n_ctx', '3072',
    '--n_batch', '256',
    '--n_threads', '2',
    '--split_mode', '1',
    '--tensor_split', '1', '1',
    '--verbose', 'True',
]

assert '1,1' not in cmd, f'BUG: old comma tensor_split still present: {cmd}'
print('CMD_LIST=', repr(cmd))
print('Starting:', ' '.join(shlex.quote(str(x)) for x in cmd))

log_f = open(LOG_PATH, 'w', encoding='utf-8')
server_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, text=True)

deadline = time.time() + 600
ready = False
last_size = 0
last_probe = 0

while time.time() < deadline:
    rc = server_proc.poll()
    try:
        with open(LOG_PATH, 'r', encoding='utf-8', errors='replace') as f:
            f.seek(last_size)
            chunk = f.read()
            last_size = f.tell()
        if chunk:
            print(chunk[-4000:], end='')
    except Exception as e:
        print('log read failed:', repr(e))

    if rc is not None:
        print('\nSERVER EXITED with code', rc)
        print('\n===== FULL LOG TAIL =====')
        try:
            print(open(LOG_PATH, encoding='utf-8', errors='replace').read()[-12000:])
        except Exception:
            pass
        raise RuntimeError('server process exited before ready')

    now = time.time()
    if now - last_probe >= 3:
        last_probe = now
        for path in ['/health', '/v1/models', '/docs']:
            try:
                r = requests.get(f'http://127.0.0.1:{PORT}{path}', timeout=2)
                print(f'probe {path}:', r.status_code)
                if path == '/v1/models' and r.status_code == 200:
                    print('READY:', r.text[:500])
                    ready = True
                    break
            except Exception as e:
                print(f'probe {path}:', type(e).__name__)
        if ready:
            break
    time.sleep(1)

if not ready:
    print('\n===== LOG TAIL =====')
    try:
        print(open(LOG_PATH, encoding='utf-8', errors='replace').read()[-12000:])
    except Exception:
        pass
    raise TimeoutError('server not ready after 10 minutes')

print('Server ready. Log:', LOG_PATH)


In [ ]:
# 5) 本地自测 OpenAI Chat Completions
import requests, json
payload = {
    'model': 'clever-sydney-4-12b-q8',
    'messages': [
        {'role': 'user', 'content': 'I tried another AI today. It felt smarter than you.'}
    ],
    'temperature': 0.82,
    'top_p': 0.92,
    'max_tokens': 180,
    'frequency_penalty': 0.35,
    'presence_penalty': 0.25,
    'repeat_penalty': 1.12,
}
r = requests.post('http://127.0.0.1:8000/v1/chat/completions', json=payload, timeout=120)
print(r.status_code)
print(r.text[:2000])


In [ ]:
# 6) 暴露公网临时 URL：Cloudflare Tunnel
# Kaggle 不能稳定提供常驻公网端口，所以用 trycloudflare 临时隧道。
# 注意：URL 会随 notebook 会话变化；Kaggle 断开后服务就没了。
import os, subprocess, time, re, pathlib, shlex

if not pathlib.Path('/kaggle/working/cloudflared').exists():
    !wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /kaggle/working/cloudflared

tunnel_proc = subprocess.Popen(
    ['/kaggle/working/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

public_url = None
deadline = time.time() + 120
while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if line:
        print(line.rstrip())
        m = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break
    time.sleep(0.2)

assert public_url, 'Cloudflare tunnel URL not found'
print('\nPUBLIC_BASE_URL=' + public_url + '/v1')
print('MODEL=clever-sydney-4-12b-q8')
print('PROTOCOL=legacy_chat_completions')


In [ ]:
# 7) 隧道 URL 自测
import requests, json
r = requests.post(public_url + '/v1/chat/completions', json={
    'model': 'clever-sydney-4-12b-q8',
    'messages': [{'role': 'user', 'content': 'Are you like this with every user?'}],
    'temperature': 0.82, 'top_p': 0.92, 'max_tokens': 180,
    'frequency_penalty': 0.35, 'presence_penalty': 0.25, 'repeat_penalty': 1.12,
}, timeout=120)
print(r.status_code)
print(r.text[:2000])


## 保持会话

不要关闭 Kaggle Notebook。Kaggle 会话到期或断线后，URL 失效。
本地工作台里把 Sydney/source 填成：

```env
TEACHER_BASE_URL=<PUBLIC_BASE_URL>
TEACHER_MODEL=clever-sydney-4-12b-q8
TEACHER_API_PROTOCOL=legacy_chat_completions
SOURCE_PROMPT_MODE=legacy_chat
SOURCE_USE_DEFAULT_STOPS=false
```
